# ДЗ 14. NumPy: векторизация, broadcasting, оси, маски 🔢
Решите задачи в Google Colab и сдайте через GitHub: папка `hw14` в репозитории `cyberformula3-homework`.

- **[База]** — обязательно для всех ·
- **[Челлендж]** — сколько получится · 
- **[Про]** — для тех, кому легко · 

Каждый ответ — **число + одна фраза**: что это значит для магазина.
Циклы `for` по заказам для расчётов не используем — только массивы и маски. Цикл можно там, где это сказано.

**Проверь себя перед сдачей:**
□ **Среда выполнения → Перезапустить и выполнить все** — ноутбук проходит без ошибок
□ Ответы совпадают с «Примером проверки», где он есть

In [1]:
import numpy as np
# проверка: вставьте сюда строки 1–5 и запустите

## Загрузка данных — запустите как есть

In [2]:
# Загрузка: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
FILE = "orders.csv"
print("Работаем с файлом:", FILE)

Работаем с файлом: orders.csv


In [3]:
import csv
import json
import numpy as np

with open(FILE, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

price, qty, city, category, status, month, product = [], [], [], [], [], [], []
for r in rows:
    price.append(int(r["price"]))
    qty.append(int(r["qty"]))
    city.append(r["city"])
    category.append(r["category"])
    status.append(r["status"])
    month.append(int(r["date"].split("-")[1]))
    product.append(r["product"])

price = np.array(price)
qty = np.array(qty)
city = np.array(city)
category = np.array(category)
status = np.array(status)
month = np.array(month)
product = np.array(product)

revenue = price * qty
paid = status == "оплачен"
print(revenue.shape, revenue[:5])

(500,) [198000  24000 159000 189000 129000]


## [База]

### Задача 1. Цены выросли на 5%
Магазин поднимает все цены на 5%. Посчитайте новые суммы заказов **одной строкой**, без цикла.
На сколько тенге вырастет общая выручка?

**Пример проверки:** выручка вырастет на `4 625 325 ₸`.

In [4]:
new_revenue = revenue * 1.05

growth = round(new_revenue.sum() - revenue.sum())
print("Выручка вырастет на", growth)

Выручка вырастет на 4625325


**Ответ:** общая выручка вырастет на 4 625 325 ₸ (ровно на 5% от нынешней). Для магазина это дополнительный доход при тех же объёмах продаж, если покупатели не уйдут из-за повышения цен.

### Задача 2. Смартфоны
С помощью маски по `category` найдите:
- сколько заказов в категории «Смартфоны»;
- на какую сумму;
- какая это доля от всех заказов.

**Пример проверки:** `131` заказ на `30 689 000 ₸`, доля `26.2%`.

In [5]:
is_phone = category == "Смартфоны"

n_phones = int(is_phone.sum())
phones_sum = int(revenue[is_phone].sum())
phones_share = is_phone.mean()

print("Заказов:", n_phones)
print("Сумма:", phones_sum)
print("Доля:", f"{phones_share:.1%}")

Заказов: 131
Сумма: 30689000
Доля: 26.2%


**Ответ:** 131 заказ на 30 689 000 ₸, это 26,2% от всех заказов. Смартфоны составляют четверть заказов магазина, поэтому по ним нужно держать товарный запас в первую очередь.

### Задача 3. Два условия
Сколько **оплаченных** заказов из **Шымкента** и на какую сумму? Не забудьте скобки вокруг каждого условия.

**Пример проверки:** `71` заказ на `12 852 500 ₸`.

In [6]:
mask = (city == "Шымкент") & paid

n_shymkent = int(mask.sum())
sum_shymkent = int(revenue[mask].sum())

print("Заказов:", n_shymkent)
print("Сумма:", sum_shymkent)

Заказов: 71
Сумма: 12852500


**Ответ:** 71 оплаченный заказ из Шымкента на 12 852 500 ₸. Это выручка, которая действительно пришла из города, отменённые заказы в неё не входят.

### Задача 4. Неделя курьера
Первые 30 заказов — это 6 дней по 5 заказов: `w = revenue[:30].reshape(6, 5)`.
1. Сумма за каждый день — какую `axis` взять? Сколько чисел должно получиться?
2. В какой день (номер с 1) сумма самая большая?
3. Самый дорогой заказ **каждого** дня.

**Пример проверки:** лучший — **3-й** день, `1 094 000 ₸`; самые дорогие по дням — `[198000 319000 549000 449000 449000 319000]`.

In [7]:
w = revenue[:30].reshape(6, 5)

# 1. сумма за каждый день: axis=1 суммирует вдоль строки, получится 6 чисел
day_sums = w.sum(axis=1)
print(day_sums)

# 2. день с самой большой суммой (номер с 1)
best_day = day_sums.argmax() + 1
print("Лучший день:", best_day, day_sums.max())

# 3. самый дорогой заказ каждого дня
print(w.max(axis=1))

[ 699000  925000 1094000  636000 1047000  571000]
Лучший день: 3 1094000
[198000 319000 549000 449000 449000 319000]


**Ответ:** сумму за день даёт `axis=1` (суммирование вдоль каждой строки), получается 6 чисел. Лучший день третий, 1 094 000 ₸, самые дорогие заказы по дням: 198 000, 319 000, 549 000, 449 000, 449 000 и 319 000 ₸. В третий день курьер везёт товар на самую большую сумму.